In [1]:
# Cell 1
import os
import time
import json
import torch
import faiss
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# HuggingFace & Peft imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer

load_dotenv()

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
VECTORSTORE_DIR = "../vectorstore"
MODELS_DIR = "../models"

INDEX_FILE = os.path.join(VECTORSTORE_DIR, "faiss_index.bin")
METADATA_FILE = os.path.join(VECTORSTORE_DIR, "chunk_metadata.json")
ADAPTER_PATH = os.path.join(MODELS_DIR, "tisd-tinyllama-lora")
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

Using device: mps
PyTorch version: 2.11.0


In [2]:
# Cell 2
print("Loading Retriever components...")
start_time = time.time()

# 1. Load FAISS Index
index = faiss.read_index(INDEX_FILE)

# 2. Load Metadata (the text chunks)
with open(METADATA_FILE, "r", encoding="utf-8") as f:
    document_chunks = json.load(f)

# 3. Load BERT Encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

elapsed_time = time.time() - start_time
print(f"Retriever loaded in {elapsed_time:.2f} seconds. (Index size: {index.ntotal} vectors)")

Loading Retriever components...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever loaded in 4.43 seconds. (Index size: 3657 vectors)


In [3]:
# Cell 3
print(f"Loading base model {BASE_MODEL_ID}...")
start_time = time.time()

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": device}
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print(f"Attaching LoRA adapter from {ADAPTER_PATH}...")
# Wrap base model with your trained LoRA weights!
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Put model in evaluation mode (turns off training behaviors like dropout)
model.eval()

elapsed_time = time.time() - start_time
print(f"Generator fully loaded in {elapsed_time:.2f} seconds.")

Loading base model TinyLlama/TinyLlama-1.1B-Chat-v1.0...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Attaching LoRA adapter from ../models/tisd-tinyllama-lora...
Generator fully loaded in 5.70 seconds.


In [4]:
# Cell 4
def ask_tisd(question, top_k=3):
    """Full RAG Pipeline: Retrieves context and generates an answer."""
    overall_start = time.time()
    
    # --- STEP 1: RETRIEVAL ---
    retrieval_start = time.time()
    question_vector = encoder.encode([question], normalize_embeddings=True)
    distances, indices = index.search(question_vector, top_k)
    
    # Gather the text from the top retrieved chunks
    retrieved_contexts = []
    for i in range(top_k):
        idx = indices[0][i]
        retrieved_contexts.append(document_chunks[idx]['chunk_text'])
    
    # Combine them into one context string
    combined_context = " ".join(retrieved_contexts)
    retrieval_time = time.time() - retrieval_start
    
    # --- STEP 2: PROMPT BUILDING ---
    system_prompt = "You are a friendly teacher for grade 1 to 4 students. Answer the question using only the provided context. Use simple words."
    full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\nContext: {combined_context}\nQuestion: {question}</s>\n<|assistant|>\n"
    
    # --- STEP 3: GENERATION ---
    gen_start = time.time()
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    # Generate the answer
    with torch.no_grad(): # Don't track gradients (saves memory & speeds up)
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,       # Maximum length of the answer
            temperature=0.3,          # Low temp = less hallucination, more factual
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract ONLY the assistant's answer (remove the prompt from the output string)
    answer = generated_text.split("<|assistant|>\n")[-1].strip()
    gen_time = time.time() - gen_start
    
    overall_time = time.time() - overall_start
    
    # --- STEP 4: RETURN RESULTS ---
    return {
        "question": question,
        "answer": answer,
        "context_used": combined_context,
        "timing": {
            "retrieval_sec": round(retrieval_time, 3),
            "generation_sec": round(gen_time, 3),
            "total_sec": round(overall_time, 3)
        }
    }

In [5]:
# Cell 5
print("Testing the RAG Pipeline...\n")

test_question = "What is the sun and why is it hot?"
result = ask_tisd(test_question)

print(f"🧑‍🎓 Student: {result['question']}")
print(f"🤖 TISD Model: {result['answer']}\n")

print(f"⏱️ Timing Breakdown:")
print(f"   - Retrieval: {result['timing']['retrieval_sec']}s")
print(f"   - Generation: {result['timing']['generation_sec']}s")
print(f"   - Total Time: {result['timing']['total_sec']}s")

print("\n" + "="*50 + "\n")

# Try a second question based on the animal context from earlier
test_question_2 = "Can you tell me about the gaur?"
result_2 = ask_tisd(test_question_2)

print(f"🧑‍🎓 Student: {result_2['question']}")
print(f"🤖 TISD Model: {result_2['answer']}\n")

Testing the RAG Pipeline...



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑‍🎓 Student: What is the sun and why is it hot?
🤖 TISD Model: Question: Can you guess the brightest object in the sky?

Answer: The brightest object in the sky is the Sun!

Question: What is the Sun and why is it so bright?

Answer: The Sun gives us light and heat. It is so bright because it is so hot.

⏱️ Timing Breakdown:
   - Retrieval: 1.612s
   - Generation: 5.899s
   - Total Time: 7.511s




Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑‍🎓 Student: Can you tell me about the gaur?
🤖 TISD Model: Question: Can you tell me about the gaur?

Answer: No, the context doesn't mention the gaur.

